# Replication of Roud Song CLuters

Based on [Roud Song Clusters](https://github.com/sian0x0/Roud-Song-Clusters)
This uses our second dataset, dataset-v2.csv.

In [ ]:
#yyoyoyoyoyoyoyoyoyoyoyoyoyo
#hi im here
from google.colab import drive
drive.mount('/content/drive')

MessageError: Error: credential propagation was unsuccessful

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

from sentence_transformers import SentenceTransformer
from sklearn.manifold import TSNE
import hdbscan

import plotly.express as px
import plotly.io as pio

import warnings
warnings.filterwarnings("ignore")

SEED = 1337  # for determinism

In [ ]:
# Load the data, drop duplicates and any songs without lyrics
raw = pd.read_csv("dataset-v2.csv", dtype=str)

df_lyrics2 = pd.DataFrame({
    "name":    raw["title"],
    "roud":    raw["roud"],   # the answer we're evaluating
    "variant": raw["title"] + "# " + raw["version"],
    "lyrics":  raw["lyrics"],
})

df_lyrics2 = df_lyrics2.dropna(subset=["lyrics", "roud"])
df_lyrics2 = df_lyrics2.drop_duplicates(subset=["name", "lyrics", "roud"]).reset_index(drop=True)

print(len(df_lyrics2), "versions of", df_lyrics2.roud.nunique(), "songs")
df_lyrics2.head()

FileNotFoundError: [Errno 2] No such file or directory: 'dataset-v2.csv'

In [ ]:
# Keep ballads of three versions, since a cluster needs several members
df_lyrics2["roud_count"] = df_lyrics2.groupby("roud")["roud"].transform("count")
df_roud_multiples = df_lyrics2[df_lyrics2["roud_count"] >= 3]
print(len(df_roud_multiples), "variants of", df_roud_multiples.roud.nunique(), "ballads kept")

In [ ]:
# embedding step
CACHE = Path("ballad_embeddings.npy")
df = df_lyrics2.copy()
lyrics = df["lyrics"].tolist()

if CACHE.exists() and len(np.load(CACHE)) == len(lyrics):
    lyrics_embeddings = np.load(CACHE)
else:
    model = SentenceTransformer("BAAI/bge-m3")
    lyrics_embeddings = model.encode(lyrics, batch_size=4, show_progress_bar=True)
    np.save(CACHE, lyrics_embeddings)

df["lyric_embed_instructor"] = list(lyrics_embeddings)
df_lyrics_w_embeddings = df.copy()
print("embedding matrix:", lyrics_embeddings.shape, "one row per song")

In [ ]:
# cosine similarity: closer to 1.0 = same direction, lower = less alike
def cos(a, b):
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

# two versions of the same ballad vs. two different ballads
jug1, jug2 = df[df.roud == "1808"].index[:2]
fisherman  = df[df.roud == "291"].index[0]

print("same ballad     :", round(cos(lyrics_embeddings[jug1], lyrics_embeddings[jug2]), 3))
print("different ballad:", round(cos(lyrics_embeddings[jug1], lyrics_embeddings[fisherman]), 3))

# same check over the whole dataset: compare every pair of songs
roud_numbers = df.roud.tolist()
same_total, same_count = 0, 0
diff_total, diff_count = 0, 0

for i in range(len(df)):
    for j in range(i + 1, len(df)):          # j > i so each pair is counted once
        sim = cos(lyrics_embeddings[i], lyrics_embeddings[j])
        if roud_numbers[i] == roud_numbers[j]:
            same_total += sim
            same_count += 1
        else:
            diff_total += sim
            diff_count += 1

print(f"\naverage similarity, same ballad      : {same_total / same_count:.3f}")
print(f"average similarity, different ballads: {diff_total / diff_count:.3f}")

In [ ]:
# check number of dimensions in the output
lyrics_embeddings.shape


In [ ]:
#t-sne, turning many dimensions into two

TSNE_CACHE = Path("tsne_bge_m3.npy")

if TSNE_CACHE.exists() and len(np.load(TSNE_CACHE)) == len(X):
    X_tsne = np.load(TSNE_CACHE)
else:
    X_tsne = TSNE(n_components=2, random_state=SEED).fit_transform(np.asarray(X))
    np.save(TSNE_CACHE, X_tsne)

df_multi_roud_w_embed = df_lyrics_w_embeddings[df_lyrics_w_embeddings.index.isin(df_roud_multiples.index)].copy()
X = df_multi_roud_w_embed["lyric_embed_instructor"].tolist()

X_tsne = TSNE(n_components=3, random_state=SEED).fit_transform(np.asarray(X))
df_multi_roud_w_embed["tsne"] = X_tsne.tolist()
df_multi_roud_w_embed[["name", "roud", "tsne"]].head()

In [ ]:
#hdbscan, turn datapoints into clusters
hdb_tsne = hdbscan.HDBSCAN(prediction_data=True, min_cluster_size=3, min_samples=2, gen_min_span_tree=True)
data = df_multi_roud_w_embed.tsne.to_list()
hdb_tsne.fit(data)

df_multi_roud_w_embed["cluster_tsne"] = hdb_tsne.labels_
df_multi_roud_w_embed["probability_tsne"] = hdb_tsne.probabilities_

labels = hdb_tsne.labels_
print("clusters found :", labels.max() + 1, " (true number of ballads:", df_multi_roud_w_embed.roud.nunique(), ")")
print(f"noise points   : {(labels == -1).sum()} ({(labels == -1).mean():.0%})")


- **Pure cluster** = every song in it is the same ballad. This is good.
- **Mixed cluster** = contains 2+ different ballads, so the computer merged songs a human kept apart.
- **Split ballad** = one ballad spread over 2+ clusters, so the computer separated songs a human grouped.
- **Adjusted Rand Index (ARI):** 1.0 means perfect agreement with the human labels and 0 means random guessing.

In [ ]:
# how good is the clustering?
from sklearn.metrics import adjusted_rand_score

clustered = df_multi_roud_w_embed[df_multi_roud_w_embed.cluster_tsne != -1]
per_cluster = clustered.groupby("cluster_tsne")["roud"].nunique()
per_ballad  = clustered.groupby("roud")["cluster_tsne"].nunique()

print(f"pure clusters : {(per_cluster == 1).sum()} / {len(per_cluster)}")
print(f"mixed clusters: {(per_cluster > 1).sum()}")
print(f"split ballads : {(per_ballad > 1).sum()} / {len(per_ballad)}")
# ARI: give each noise point its own singleton cluster. Otherwise sklearn treats all
# "-1" points as one big (wrong) cluster and the score is unfairly low.
lab = df_multi_roud_w_embed.cluster_tsne.to_numpy().copy()
noise = lab == -1
lab[noise] = lab.max() + 1 + np.arange(noise.sum())
print(f"ARI vs Child numbers: {adjusted_rand_score(df_multi_roud_w_embed.roud, lab):.2f}")

In [ ]:
# chart/plot
df_plotly = df_multi_roud_w_embed.copy()
df_plotly["cluster_tsne"] = df_plotly["cluster_tsne"].astype(str)

# make non-cluster -1 grey
color_map = {"-1": "lightgray"}  # nb key is a string
# colours for other cluster labels
num_clusters = df_plotly["cluster_tsne"].nunique()
palette = px.colors.qualitative.Light24
color_map.update({str(i): palette[i % len(palette)] for i in range(num_clusters - 1)})

# calculate localized centroids for 'roud' annotations
def calculate_annotations():
    unique_roud_values = df_plotly["roud"].unique()
    annotations = []

    for roud_value in unique_roud_values:
        roud_clusters = df_plotly[df_plotly["roud"] == roud_value]["cluster_tsne"].unique()

        for cluster_label in roud_clusters:
            roud_points = df_plotly[(df_plotly["roud"] == roud_value) & (df_plotly["cluster_tsne"] == cluster_label)]
            if len(roud_points) > 0:
                centroid_x = roud_points["tsne"].str.get(0).mean()
                centroid_y = roud_points["tsne"].str.get(1).mean()
                if len(roud_points) > 1:
                    annotations.append(dict(x=centroid_x, y=centroid_y, xref="x", yref="y",
                                            text=roud_value, showarrow=True, arrowhead=2, ax=0, ay=-25))
                else:
                    annotations.append(dict(x=centroid_x, y=centroid_y, xref="x", yref="y",
                                            text=roud_value, showarrow=False, ax=0, ay=0))
    return annotations

annotations = calculate_annotations()

# plot
fig = px.scatter(df_plotly,
                 x=df_plotly["tsne"].str.get(0),
                 y=df_plotly["tsne"].str.get(1),
                 color="cluster_tsne",
                 color_discrete_map=color_map,
                 hover_name="name",
                 hover_data={"roud": True, "variant": True},
                 labels={"cluster_tsne": "Cluster Label", "roud": "Child no."},
                 title="Child ballads: t-SNE reduced HDBSCAN Clustering (labels = true Child number)")

button_on  = dict(label="Annotations On",  method="relayout", args=[{"annotations": annotations}])
button_off = dict(label="Annotations Off", method="relayout", args=[{"annotations": []}])
fig.update_layout(updatemenus=[dict(type="buttons", showactive=True, buttons=[button_on, button_off])],
                  annotations=annotations)

fig.update_layout(xaxis_title="t-SNE X", yaxis_title="t-SNE Y", showlegend=False)
fig.update_xaxes(scaleanchor="y", scaleratio=1)
fig.update_yaxes(scaleanchor="x", scaleratio=1)
fig.update_layout(width=1400, height=1100)

fig.write_html("ballad_clusters.html")
fig.show()

Each ballad's lyrics were turned into 1024 numbers by an embedding language model, which places songs with similar wording and
themes near each other. t-SNE squashed those numbers into 2 so they can be drawn and so that dense neighborhoods stand
out. HDBSCAN then circled the dense neighbourhoods without being told how many to find, and it set aside the songs it
couldn't place. Finally we compared the circles with Child's human-made groups. About two-thirds of the clusters are
**pure** (one ballad each), and the overall agreement is decent but far from perfect with about a .7